<a href="https://colab.research.google.com/github/pedroManuelP/C-digos-em-Python/blob/main/Lista2_do_Projeto_de_Sistemas_de_Controle.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np

def coefAmort(Mp_percent):
    n = np.log(Mp_percent/100)
    d = np.sqrt(np.power(np.pi,2) + np.power(np.log(Mp_percent/100),2))
    return n/d

def freqNatural(coefAmort,tss, tss_percent):
  if(tss_percent == 5):
    wn = 3/(tss*coefAmort)
  elif(tss_percent == 2):
    wn = wn = 4/(tss*coefAmort)
  return wn

def anguloThetaPhi(si, p):
  if(si.real > p.real):
    angulo = np.atan2((si.imag - p.imag), (p.real - si.real))
    angulo = 180 - np.rad2deg(angulo)
  elif(si.real < p.real):
    angulo = np.atan2((si.imag - p.imag), (si.real - p.real))
    angulo = np.rad2deg(angulo)
  else:
    angulo = 90
  return angulo

In [3]:
kg = 5; g_num = kg*np.polynomial.Polynomial([3, 1]); g_den = np.polynomial.Polynomial([0, 4, 1])
print("G(s) = (" + str(g_num) + ")/"); print("(" + str(g_den) + ")")

kh = 1; h_num = kh*np.polynomial.Polynomial([1]); h_den = np.polynomial.Polynomial([1, 1])
print("\nH(s) = (" + str(h_num) + ")/"); print("(" + str(h_den) + ")")

G(s) = (15.0 + 5.0·x)/
(0.0 + 4.0·x + 1.0·x²)

H(s) = (1.0)/
(1.0 + 1.0·x)


In [4]:
Mp_percent = 10; tss = 3; tss_percent = 5

ksi = np.abs(coefAmort(Mp_percent)); ksi = np.round(ksi,4)
wn = freqNatural(ksi,tss,tss_percent); wn = np.round(wn,4)
print("Coeficiente de amortecimento: " + str(ksi))
print("Frequência natural: " + str(wn))

Coeficiente de amortecimento: 0.5912
Frequência natural: 1.6915


In [5]:
wd = wn*np.sqrt(1-np.power(ksi,2)); wd = np.round(wd,4)
print("Frequência amortecida: " + str(wd))

s1 = np.round(complex(-ksi*wn,wd), 4); s2 = np.round(complex(-ksi*wn,-wd), 4)

Frequência amortecida: 1.3642


In [6]:
print("s1: " + str(s1))
print("s2: " + str(s2))

s1: (-1+1.3642j)
s2: (-1-1.3642j)


In [ ]:
# Controlador PD

polos = (g_den*h_den).roots(); zeros = (g_num*h_num).roots()  # polos e zeros da FT de malha fechada do sistema
#print(polos); print(zeros)

somatorioTheta = 0; somatorioPhi = 0
for i in range(len(polos)):
  somatorioTheta += anguloThetaPhi(s1, polos[i])
for i in range(len(zeros)):
  somatorioPhi += anguloThetaPhi(s1, zeros[i])
phiZ = somatorioTheta - somatorioPhi - 180
#z = -s1.real + s1.imag/np.tan(np.deg2rad(phiZ))                          # para um zero a esquerda de si
z = -s1.real - s1.imag/np.tan(np.deg2rad(180 - phiZ))                   # para um zero a direita de si
zeros = np.append(zeros, -z); print("Zero(s) do controlador projetado: " + str(round(z, 4)))

produtorioA = 1; produtorioB = 1
for i in range(len(polos)):
  produtorioA *= abs(s1 - (polos[i]))
for i in range(len(zeros)):
  produtorioB *= abs(s1 - (zeros[i]))
kt = produtorioA/produtorioB
kc = kt/(kg*kh)
print("Ganho do controlador projetado: " + str(round(kc, 4)))

kd = kc; kp = kd*z
print("\nGanho proporcional do controlador: " + str(round(kp, 4)))
print("Ganho derivativo do controlador: " + str(round(kd, 4)))

Zero(s) do controlador projetado: 1.3389
Ganho do controlador projetado: 11492.984

Ganho proporcional do controlador: 15387.7988
Ganho derivativo do controlador: 11492.984


In [ ]:
# Controlador PI

polos = (g_den*h_den).roots(); zeros = (g_num*h_num).roots()  # polos e zeros da FT de malha fechada do sistema
polos = np.append(polos, 0); #print(polos); print(zeros)

somatorioTheta = 0; somatorioPhi = 0
for i in range(len(polos)):
  somatorioTheta += anguloThetaPhi(s1, polos[i])
for i in range(len(zeros)):
  somatorioPhi += anguloThetaPhi(s1, zeros[i])
phiZ = somatorioTheta - somatorioPhi - 180
#z = -s1.real + s1.imag/np.tan(np.deg2rad(phiZ))                          # para um zero a esquerda de si
z = -s1.real - s1.imag/np.tan(np.deg2rad(180 - phiZ))                   # para um zero a direita de si
zeros = np.append(zeros, -z); print("Zero(s) do controlador projetado: " + str(round(z, 4)))

produtorioA = 1; produtorioB = 1
for i in range(len(polos)):
  produtorioA *= abs(s1 - (polos[i]))
for i in range(len(zeros)):
  produtorioB *= abs(s1 - (zeros[i]))
kt = produtorioA/produtorioB
kc = kt/(kg*kh)
print("Ganho do controlador projetado: " + str(round(kc, 4)))

kp = kc; ki = kp*z
print("\nGanho proporcional do controlador: " + str(round(kp, 4)))
print("Ganho integrativo do controlador: " + str(round(ki, 4)))

Zero(s) do controlador projetado: 3.4286
Ganho do controlador projetado: 7.0

Ganho proporcional do controlador: 7.0
Ganho integrativo do controlador: 24.0


In [7]:
# Controlador PID

polos = (g_den*h_den).roots(); zeros = (g_num*h_num).roots()  # polos e zeros da FT de malha fechada do sistema
polos = np.append(polos, 0); #print(polos); print(zeros)

somatorioTheta = 0; somatorioPhi = 0
for i in range(len(polos)):
  somatorioTheta += anguloThetaPhi(s1, polos[i])
for i in range(len(zeros)):
  somatorioPhi += anguloThetaPhi(s1, zeros[i])
phiZ = (somatorioTheta - somatorioPhi - 180)/2
z = -s1.real + s1.imag/np.tan(np.deg2rad(phiZ))                          # para um zero a esquerda de si
#z = -s1.real - s1.imag/np.tan(np.deg2rad(180 - phiZ))                   # para um zero a direita de si
zeros = np.append(zeros, [-z, -z]); print("Zero(s) do controlador projetado: " + str(round(z, 4)))

produtorioA = 1; produtorioB = 1
for i in range(len(polos)):
  produtorioA *= abs(s1 - (polos[i]))
for i in range(len(zeros)):
  produtorioB *= abs(s1 - (zeros[i]))
kt = produtorioA/produtorioB
kc = kt/(kg*kh)
print("Ganho do controlador projetado: " + str(round(kc, 4)))

kd = kc; kp = 2*z*kd; ki = kd*np.power(z,2);
print("\nGanho proporcional do controlador: " + str(round(kp, 4)))
print("Ganho integrativo do controlador: " + str(round(ki, 4)))
print("Ganho derivativo do controlador: " + str(round(kd, 4)))

Zero(s) do controlador projetado: 1.3321
Ganho do controlador projetado: 0.539

Ganho proporcional do controlador: 1.4361
Ganho integrativo do controlador: 0.9565
Ganho derivativo do controlador: 0.539
